In [2]:
import shutil
import pandas as pd
from pathlib import Path

In [3]:
# Caminhos
ORIGEM = Path("/media/user2/0b7e9258-fafd-4f97-8bf0-5b86acf8e9b4/CBIS-DDSM/cbis_ddsm_jpg")
DESTINO = Path("/media/user2/0b7e9258-fafd-4f97-8bf0-5b86acf8e9b4/CBIS-DDSM/segmentacao_base")
CSV_DIR = Path("/media/user2/0b7e9258-fafd-4f97-8bf0-5b86acf8e9b4/CBIS-DDSM/csv")

### 1: Auditoria e filtragem

Contagem de imagens e pacientes para validar os números com a análise exploratória. 

A verificação inicial identifica a disponibilidade de mamografias completas e ROIs por paciente. <br>
Em seguida, o script filtra apenas as mamografias completas que possuem ROI associada e copia os arquivos aprovados para a nova base de segmentação.

In [10]:
pastas_origem = [p for p in ORIGEM.iterdir() if p.is_dir() and ('Mass-' in p.name or 'Calc-' in p.name)]

completas = {}
rois = {}

for pasta in pastas_origem:
    if pasta.name[-1].isdigit():
        # É uma ROI. Pega o nome base removendo o último _numero
        base = '_'.join(pasta.name.split('_')[:-1])
        rois.setdefault(base, []).append(pasta.name)
    else:
        completas[pasta.name] = pasta.name

# Extrair o ID do paciente a partir do nome da pasta
def extrair_paciente(nome_pasta):
    partes = nome_pasta.split('_P_')
    if len(partes) > 1:
        return 'P_' + partes[1].split('_')[0]
    return None

# Contagem de Pacientes
pacientes_completas = set(extrair_paciente(nome) for nome in completas.keys())
pacientes_rois = set(extrair_paciente(nome) for nome in rois.keys())

pacientes_ambos = pacientes_completas & pacientes_rois
pacientes_so_completa = pacientes_completas - pacientes_rois
pacientes_so_roi = pacientes_rois - pacientes_completas

print("AUDITORIA POR PACIENTES (Validação com Análise Exploratória)")
print(f"Pacientes com mamografia completa: {len(pacientes_completas)}")
print(f"Pacientes com ROI:                 {len(pacientes_rois)}")
print(f"Pacientes com ambos:               {len(pacientes_ambos)}")
print(f"Pacientes apenas com completa:     {len(pacientes_so_completa)}")
print(f"Pacientes apenas com ROI (órfãos): {len(pacientes_so_roi)}")

# Contagem de Imagens (Pastas)
completas_validas = []
completas_sem_roi = []
rois_orfas = 0

for nome_completa in completas.keys():
    if nome_completa in rois:
        completas_validas.append(nome_completa)
    else:
        completas_sem_roi.append(nome_completa)

for nome_base in rois.keys():
    if nome_base not in completas:
        rois_orfas += len(rois[nome_base])

print("\nAUDITORIA POR IMAGENS")
print(f"Total de pastas de mamografias completas: {len(completas)}")
print(f"Total de pastas de ROIs:                  {sum(len(v) for v in rois.values())}")
print(f"Mamografias completas válidas (com ROI):  {len(completas_validas)}")
print(f"Mamografias completas sem ROI:            {len(completas_sem_roi)}")
print(f"ROIs órfãs (sem mamografia completa):     {rois_orfas}")

AUDITORIA POR PACIENTES (Validação com Análise Exploratória)
Pacientes com mamografia completa: 1460
Pacientes com ROI:                 1530
Pacientes com ambos:               1452
Pacientes apenas com completa:     8
Pacientes apenas com ROI (órfãos): 78

AUDITORIA POR IMAGENS
Total de pastas de mamografias completas: 2857
Total de pastas de ROIs:                  3411
Mamografias completas válidas (com ROI):  2839
Mamografias completas sem ROI:            18
ROIs órfãs (sem mamografia completa):     146


In [11]:
copiadas = 0

# Copia os mamogramas válidos e suas ROIs
for nome_completa in completas_validas:
    caminho_origem_completa = ORIGEM / nome_completa
    shutil.copytree(str(caminho_origem_completa), str(DESTINO / nome_completa), dirs_exist_ok=True)
    copiadas += 1
    
    # Copia todas as ROIs associadas
    for nome_roi in rois[nome_completa]:
        caminho_origem_roi = ORIGEM / nome_roi
        shutil.copytree(str(caminho_origem_roi), str(DESTINO / nome_roi), dirs_exist_ok=True)
        copiadas += 1

print(f"Total de pastas copiadas para segmentacao_base: {copiadas}")

Total de pastas copiadas para segmentacao_base: 6104


### 2: Tratamento de Data Leakage

Identificação de mamogramas completos duplicados entre os conjuntos de Treino e Teste usando a chave única composta por: ID do Paciente + Lado + Vista (ex: P_00001_LEFT_CC). <br>
Caso um paciente tenha, por exemplo, o mamograma de vista CC no treino e o de vista MLO em training, não é considerado vazamento de dados. 

Caso uma imagem seja encontrada em ambos os conjuntos, ela é removida do conjunto de Teste para garantir a integridade da avaliação.

In [4]:
CSV_DIR = Path("/media/user2/0b7e9258-fafd-4f97-8bf0-5b86acf8e9b4/CBIS-DDSM/csv")

calc_train = pd.read_csv(CSV_DIR / "calc_case_description_train_set.csv")
calc_test  = pd.read_csv(CSV_DIR / "calc_case_description_test_set.csv")
mass_train = pd.read_csv(CSV_DIR / "mass_case_description_train_set.csv")
mass_test  = pd.read_csv(CSV_DIR / "mass_case_description_test_set.csv")

for df, nome_set in [(calc_train, 'Training'), (calc_test, 'Test'), 
                     (mass_train, 'Training'), (mass_test, 'Test')]:
    df['set'] = nome_set

df = pd.concat([calc_train, calc_test, mass_train, mass_test], ignore_index=True)

# Criar a chave única da imagem (Paciente + Lado + Vista)
# Exemplo: P_00001_LEFT_CC
df['chave_imagem'] = df['patient_id'] + '_' + df['left or right breast'] + '_' + df['image view']

# Identificar chaves presentes no Treino e no Teste simultaneamente
chaves_treino = set(df[df['set'] == 'Training']['chave_imagem'])
chaves_teste = set(df[df['set'] == 'Test']['chave_imagem'])

vazamentos = chaves_treino.intersection(chaves_teste)
print(f"Imagens duplicadas detectadas (Treino ∩ Teste): {len(vazamentos)}")

removidos_imagens = 0
removidos_pacientes = set()

if len(vazamentos) > 0:
    for chave in vazamentos:
        # Buscar as linhas do teste que contêm essa chave
        linhas_teste = df[(df['chave_imagem'] == chave) & (df['set'] == 'Test')]
        
        for _, row in linhas_teste.iterrows():
            # Extrai o nome da pasta a partir do caminho do CSV
            caminho_completo = row['image file path'].split('/')[0]
            caminho_pasta_teste = DESTINO / caminho_completo
            
            if caminho_pasta_teste.exists():
                shutil.rmtree(caminho_pasta_teste)
                removidos_imagens += 1
                removidos_pacientes.add(row['patient_id'])

print(f"Imagens completas removidas do Teste: {removidos_imagens}")
print(f"Pacientes afetados: {len(removidos_pacientes)}")

Imagens duplicadas detectadas (Treino ∩ Teste): 29
Imagens completas removidas do Teste: 16
Pacientes afetados: 12


### 3: Limpeza Final e Consistência de Pastas

Após a remoção das imagens completas com vazamento, as ROIs associadas a essas imagens ficaram órfãs na pasta. <br>
Assim, uma nova varredura foi feita para remover qualquer ROI que não tivesse um mamograma completo associado.<br>
Também foi verificado, por segurança, o caminho inverso: remover imagens completas que, por algum motivo, tenham ficado sem ROI. 

Garante que a base final contenha apenas pares perfeitos (Imagem Completa + ROIs correspondentes).

In [5]:
# Extrair o nome da mamografia completa a partir da ROI
# Exemplo: Mass-Test_P_00001_LEFT_CC_1 -> Mass-Test_P_00001_LEFT_CC
def pegar_nome_mamografia(nome_roi):
    return '_'.join(nome_roi.split('_')[:-1])

rois_removidas = 0
completas_removidas = 0

pastas_atuais = [p for p in DESTINO.iterdir() if p.is_dir() and ('Mass-' in p.name or 'Calc-' in p.name)]

# Remove ROIs cuja mamografia completa foi deletada na etapa anterior
for pasta in pastas_atuais:
    if pasta.name[-1].isdigit(): 
        nome_mamografia = pegar_nome_mamografia(pasta.name)
        caminho_mamografia = DESTINO / nome_mamografia
        
        if not caminho_mamografia.exists():
            shutil.rmtree(pasta)
            rois_removidas += 1

# Atualiza a lista e remove mamografias que porventura ficaram sem ROI
pastas_atuais = [p for p in DESTINO.iterdir() if p.is_dir() and ('Mass-' in p.name or 'Calc-' in p.name)]

for pasta in pastas_atuais:
    if not pasta.name[-1].isdigit():
        tem_roi = any(irma.is_dir() and irma.name.startswith(pasta.name + '_') for irma in pastas_atuais)
        
        if not tem_roi:
            shutil.rmtree(pasta)
            completas_removidas += 1

print(f"ROIs removidas: {rois_removidas}")
print(f"Mamografias removidas: {completas_removidas}")

ROIs removidas: 18
Mamografias removidas: 0


### 4: Verificação Final

Contagem final por imagens e por pacientes.

Também valida a integridade da árvore de diretórios, garantindo que não há mais órfãos e que cada pasta contém o número exato de arquivos esperados (1 imagem e 1 máscara para ROIs, e 1 imagem para completas).

In [7]:
def extrair_paciente(nome_pasta):
    partes = nome_pasta.split('_P_')
    if len(partes) > 1:
        return 'P_' + partes[1].split('_')[0]
    return None

pastas_finais = [p for p in DESTINO.iterdir() if p.is_dir() and ('Mass-' in p.name or 'Calc-' in p.name)]
rois_finais = [p for p in pastas_finais if p.name[-1].isdigit()]
completas_finais = [p for p in pastas_finais if not p.name[-1].isdigit()]

pacientes_finais = set(extrair_paciente(p.name) for p in pastas_finais)

print("Contagem por pastas (imagens):")
print(f"  Mamografias completas: {len(completas_finais)}")
print(f"  ROIs: {len(rois_finais)}")
print(f"  Total: {len(pastas_finais)}")

print(f"\nContagem por pacientes:")
print(f"  Pacientes únicos: {len(pacientes_finais)}")

# Checagem de integridade
erros = 0

# Verifica ROIs sem mamografia mãe
for pasta in rois_finais:
    if not (DESTINO / pegar_nome_mamografia(pasta.name)).exists():
        erros += 1

# Verifica mamografias sem nenhuma ROI filha
for pasta in completas_finais:
    if not any(irma.name.startswith(pasta.name + '_') for irma in rois_finais):
        erros += 1

# Verifica quantidade de arquivos por pasta
for pasta in pastas_finais:
    jpgs = list(pasta.glob("*.jpg"))
    pngs = list(pasta.glob("*.png"))
    
    if pasta.name[-1].isdigit(): 
        if len(jpgs) != 1 or len(pngs) != 1:
            erros += 1
    else:
        if len(jpgs) != 1:
            erros += 1

if erros == 0:
    print("\nStatus: Base íntegra e consistente.")
else:
    print(f"\nStatus: {erros} inconsistências encontradas.")

Contagem por pastas (imagens):
  Mamografias completas: 2823
  ROIs: 3247
  Total: 6070

Contagem por pacientes:
  Pacientes únicos: 1449

Status: Base íntegra e consistente.
